

---

# 📘 LeetCode 2720: Popularity Percentage

---

## ❓ Question

We want to calculate the **popularity percentage** for each user on Meta/Facebook.  

- The popularity percentage is defined as:  

{Popularity Percentage} ={Number of friends of user}/{{Total number of users}} * 100


- Round the result to **2 decimal places**.  
- Return the result table ordered by `user1` in ascending order.  

---

## 📊 Sample Data

### Friends Table

| user1 | user2 |
|-------|-------|
| 2     | 1     |
| 1     | 3     |
| 4     | 1     |
| 1     | 5     |
| 1     | 6     |
| 2     | 6     |
| 7     | 2     |
| 8     | 3     |
| 3     | 9     |

---

### Expected Output

| user1 | percentage_popularity |
|-------|-----------------------|
| 1     | 55.56                 |
| 2     | 33.33                 |
| 3     | 33.33                 |
| 4     | 11.11                 |
| 5     | 11.11                 |
| 6     | 22.22                 |
| 7     | 11.11                 |
| 8     | 11.11                 |
| 9     | 11.11                 |

---

## 🏗️ Schema Definition

```python
from pyspark.sql.types import StructType, StructField, IntegerType

friends_schema = StructType([
    StructField("user1", IntegerType(), False),
    StructField("user2", IntegerType(), False)
])
```

---

## 📥 Data Preparation

```python
friends_data = [
    (2, 1),
    (1, 3),
    (4, 1),
    (1, 5),
    (1, 6),
    (2, 6),
    (7, 2),
    (8, 3),
    (3, 9)
]
```

---

## 🗂️ Create DataFrame

```python
friends_df = spark.createDataFrame(friends_data, schema=friends_schema)
friends_df.show()
```

---

## 👁️ Register as SQL View

```python
friends_df.createOrReplaceTempView("Friends")
```

---


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

friends_schema = StructType([
    StructField("user1", IntegerType(), False),
    StructField("user2", IntegerType(), False)
])
friends_data = [
    (2, 1),
    (1, 3),
    (4, 1),
    (1, 5),
    (1, 6),
    (2, 6),
    (7, 2),
    (8, 3),
    (3, 9)
]
friends_df = spark.createDataFrame(friends_data, schema=friends_schema)
friends_df.show()
friends_df.createOrReplaceTempView("Friends")


In [0]:
%sql
With cte as (
  select user1   self   , user2   friend     from Friends
  union 
   select user2    self  , user1  friend      from Friends
), cte2 as (
  select self , friend ,
   count(friend)over(partition by self ) as cnt_frinds  ,
   (select count(distinct self ) from cte) as total_cnt
  from cte
)
select distinct self , round(100*(cnt_frinds / total_cnt),2) from cte2

order by 1 asc